# Study 0 — Ablation Studies

Architecture ablations and training-set scaling curves.

In [ ]:
%cd /content/drive/MyDrive/Fin-JEPA/repo

# Install fin-jepa as editable WITHOUT upgrading Colab's pre-installed deps
!pip install -q --no-deps -e '.[dev]'

# Install only the deps Colab doesn't already ship
!pip install -q hydra-core mlflow optuna xgboost yfinance

import sys
# Path updated to match the location in your Google Drive
repo_path = '/content/drive/MyDrive/Fin-JEPA/repo/src'
if repo_path not in sys.path:
    sys.path.append(repo_path)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from fin_jepa.training.ablations import run_ablations

sns.set_theme(style="whitegrid", palette="colorblind")
RESULTS_DIR = Path("/content/drive/MyDrive/Fin-JEPA/repo/results/study0/ablations")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Run Ablation Sweeps

In [ ]:
REPO = "/content/drive/MyDrive/Fin-JEPA/repo"

_sweep_files = {
    "n_layers":       RESULTS_DIR / "n_layers_sweep.json",
    "d_token":        RESULTS_DIR / "d_token_sweep.json",
    "train_fraction": RESULTS_DIR / "train_fraction_sweep.json",
    "mask_ratio":     RESULTS_DIR / "mask_ratio_sweep.json",
}

all_results = {}

if all(p.exists() for p in _sweep_files.values()):
    print("Loading cached ablation results...")
    for name, path in _sweep_files.items():
        with open(path) as f:
            all_results[name] = json.load(f)
    print("Done.")
else:
    print("Running ablation sweeps — expect ~4 hours on GPU...")
    config = {
        "data": {
            "raw_dir": f"{REPO}/data/raw",
            "processed_dir": f"{REPO}/data/processed",
            "split": {
                "train_end": "2017-12-31",
                "val_end":   "2019-12-31",
                "test_end":  "2023-12-31",
            },
        },
        # Note: 'features' config is not consumed by run_ablations() — ablations.py calls
        # build_feature_matrix() with default FeatureConfig. Kept here to document the
        # feature set in use; update ablations.py if you need non-default features.
        "features": {
            "use_raw": True,
            "use_ratios": True,
            "use_yoy": True,
            "use_missingness_flags": True,
            "coverage_threshold": 0.50,
            "normalization_method": "quantile",
            "median_impute": True,
        },
        "outcomes": ["stock_decline"],
        "sweep": {
            "n_layers":        [1, 2, 3, 6],
            "d_token":         [64, 128, 192, 256],
            "train_fractions": [0.10, 0.25, 0.50, 0.75, 1.0],
            "mask_ratio":      [0.10, 0.15, 0.20, 0.30],
        },
        "results_dir": str(RESULTS_DIR),
        "seed": 42,
    }
    all_results = run_ablations(config)
    print("All sweeps complete.")

## 2. Architecture Ablations (n_layers & d_token)

One-dimensional sweeps over depth and width, holding all other params at benchmark defaults
(d_token=192, n_heads=8, n_layers=3). Dotted line marks the default value.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, sweep_name, xlabel, default_val in [
    (axes[0], "n_layers", "n_layers",  3),
    (axes[1], "d_token",  "d_token",   192),
]:
    rows = [r for r in all_results[sweep_name] if r.get("auroc") is not None]
    if not rows:
        ax.set_title(f"{sweep_name} — no results")
        continue

    df = pd.DataFrame(rows).sort_values(xlabel)
    ax.plot(df[xlabel], df["auroc"], marker="o", linewidth=2, label="AUROC")
    if "auprc" in df.columns:
        ax.plot(df[xlabel], df["auprc"], marker="s", linewidth=2, linestyle="--", label="AUPRC")
    ax.axvline(default_val, color="grey", linestyle=":", linewidth=1.5, label=f"default ({default_val})")

    best_idx = df["auroc"].idxmax()
    best_row = df.loc[best_idx]
    ax.annotate(
        f"  best={best_row['auroc']:.4f}",
        xy=(best_row[xlabel], best_row["auroc"]),
        fontsize=9, color="steelblue",
    )

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Score")
    ax.set_title(f"{xlabel} sweep (stock_decline)")
    ax.legend()

plt.tight_layout()
plt.show()

## 3. Training-Set Scaling Curve

How much training data does the model actually need? Log-scale x-axis reveals the power-law
relationship between data size and performance.

In [ ]:
rows = [r for r in all_results["train_fraction"] if r.get("auroc") is not None]
df_scale = pd.DataFrame(rows).sort_values("n_train")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_scale["n_train"], df_scale["auroc"], marker="o", linewidth=2, label="AUROC")
if "auprc" in df_scale.columns:
    ax.plot(df_scale["n_train"], df_scale["auprc"], marker="s", linewidth=2,
            linestyle="--", label="AUPRC")

# Power-law fit on log-log scale
log_n = np.log(df_scale["n_train"].values.astype(float))
log_a = np.log(df_scale["auroc"].values.astype(float))
if len(log_n) >= 2:
    alpha, log_c = np.polyfit(log_n, log_a, 1)
    n_fit = np.linspace(df_scale["n_train"].min(), df_scale["n_train"].max(), 200)
    a_fit = np.exp(log_c) * n_fit ** alpha
    ax.plot(n_fit, a_fit, linestyle=":", color="grey", label=f"power-law fit (α={alpha:.3f})")

ax.set_xscale("log")
ax.set_xlabel("Training samples (log scale)")
ax.set_ylabel("AUROC")
ax.set_title("Scaling curve — stock_decline")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nFull-data AUROC:  {df_scale['auroc'].iloc[-1]:.4f}")
half_idx = (df_scale["train_fraction"] - 0.50).abs().idxmin()
print(f"50%-data AUROC:   {df_scale.loc[half_idx, 'auroc']:.4f}")
print(f"Gap at 50%:       {df_scale['auroc'].iloc[-1] - df_scale.loc[half_idx, 'auroc']:+.4f}")

## 4. SSL Mask Ratio Ablation

Each bar is a separate pretrain (50 epochs) + fine-tune run. The dashed line is the
from-scratch baseline (no SSL). Note: the full SSL experiment (03_ssl_pretraining.ipynb)
used 200 pretrain epochs — gains here may be slightly smaller.

In [ ]:
rows_mr = [r for r in all_results["mask_ratio"] if r.get("auroc") is not None]
df_mr = pd.DataFrame(rows_mr).sort_values("mask_ratio")

# Baseline = from-scratch FT-Transformer (n_layers default sweep at default value)
_defaults = [r for r in all_results["n_layers"]
             if r.get("n_layers") == 3 and r.get("auroc") is not None]
baseline_auroc = _defaults[0]["auroc"] if _defaults else None

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    [f"MR={r['mask_ratio']:.2f}" for r in rows_mr],
    df_mr["auroc"],
    color=sns.color_palette("colorblind", len(rows_mr)),
)
if baseline_auroc is not None:
    ax.axhline(baseline_auroc, color="grey", linestyle="--", linewidth=1.5,
               label=f"from-scratch ({baseline_auroc:.4f})")
    ax.legend()

ax.set_xlabel("Mask ratio")
ax.set_ylabel("AUROC")
ax.set_title("SSL mask ratio ablation — stock_decline (50 pretrain epochs)")
for bar, val in zip(bars, df_mr["auroc"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.002, f"{val:.4f}",
            ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

best_mr_row = df_mr.loc[df_mr["auroc"].idxmax()]
print(f"Best mask ratio:  {best_mr_row['mask_ratio']} (AUROC={best_mr_row['auroc']:.4f})")
if baseline_auroc is not None:
    print(f"Delta vs scratch: {best_mr_row['auroc'] - baseline_auroc:+.4f}")

## 5. Summary & Recommendations

In [ ]:
_DEFAULT_N_LAYERS = 3
_DEFAULT_D_TOKEN  = 192

def _best(sweep_name, param_col):
    rows = [r for r in all_results[sweep_name] if r.get("auroc") is not None]
    if not rows:
        return None, None, None
    df = pd.DataFrame(rows)
    best = df.loc[df["auroc"].idxmax()]
    default_row = df[df[param_col] == (_DEFAULT_N_LAYERS if param_col == "n_layers" else _DEFAULT_D_TOKEN)]
    default_auroc = default_row["auroc"].values[0] if len(default_row) else None
    return best[param_col], best["auroc"], default_auroc

n_layers_best_val, n_layers_best_auroc, n_layers_default_auroc = _best("n_layers", "n_layers")
d_token_best_val,  d_token_best_auroc,  d_token_default_auroc  = _best("d_token",  "d_token")

_tf_rows = [r for r in all_results["train_fraction"] if r.get("auroc") is not None]
df_tf = pd.DataFrame(_tf_rows).sort_values("train_fraction") if _tf_rows else pd.DataFrame()
full_auroc = df_tf["auroc"].iloc[-1] if not df_tf.empty else None
_half_idx  = (df_tf["train_fraction"] - 0.50).abs().idxmin() if not df_tf.empty else None
half_auroc = df_tf.loc[_half_idx, "auroc"] if _half_idx is not None else None

_mr_rows = [r for r in all_results["mask_ratio"] if r.get("auroc") is not None]
df_mr2 = pd.DataFrame(_mr_rows) if _mr_rows else pd.DataFrame()
mr_best_val   = df_mr2.loc[df_mr2["auroc"].idxmax(), "mask_ratio"] if not df_mr2.empty else None
mr_best_auroc = df_mr2["auroc"].max() if not df_mr2.empty else None

# Derive baseline locally so this cell is safe to run after a kernel restart
_defaults_s5 = [r for r in all_results["n_layers"]
                if r.get("n_layers") == 3 and r.get("auroc") is not None]
baseline_auroc = _defaults_s5[0]["auroc"] if _defaults_s5 else None

summary_rows = [
    {"Sweep":       "n_layers",
     "Best value":  n_layers_best_val,
     "Best AUROC":  f"{n_layers_best_auroc:.4f}" if n_layers_best_auroc else "—",
     "Default AUROC": f"{n_layers_default_auroc:.4f}" if n_layers_default_auroc else "—",
     "Change?":     "update" if n_layers_best_val != _DEFAULT_N_LAYERS else "keep default"},
    {"Sweep":       "d_token",
     "Best value":  d_token_best_val,
     "Best AUROC":  f"{d_token_best_auroc:.4f}"  if d_token_best_auroc  else "—",
     "Default AUROC": f"{d_token_default_auroc:.4f}"  if d_token_default_auroc  else "—",
     "Change?":     "update" if d_token_best_val != _DEFAULT_D_TOKEN else "keep default"},
    {"Sweep":       "train_fraction",
     "Best value":  "1.00",
     "Best AUROC":  f"{full_auroc:.4f}" if full_auroc else "—",
     "Default AUROC": f"{half_auroc:.4f}" if half_auroc else "—",
     "Change?":     f"gap at 50%: {full_auroc - half_auroc:+.4f}" if (full_auroc and half_auroc) else "—"},
    {"Sweep":       "mask_ratio",
     "Best value":  mr_best_val,
     "Best AUROC":  f"{mr_best_auroc:.4f}" if mr_best_auroc else "—",
     "Default AUROC": f"{baseline_auroc:.4f}" if baseline_auroc else "—",
     "Change?":     f"SSL delta: {mr_best_auroc - baseline_auroc:+.4f}" if (mr_best_auroc and baseline_auroc) else "—"},
]

print("=" * 65)
print("  Ablation summary — stock_decline")
print("=" * 65)
display(pd.DataFrame(summary_rows).set_index("Sweep"))

print("\nRecommendations for Studies 1-2:")
print(f"  • n_layers:   {'keep default (3)' if n_layers_best_val == _DEFAULT_N_LAYERS else f'consider updating to {n_layers_best_val}'}")
print(f"  • d_token:    {'keep default (192)' if d_token_best_val == _DEFAULT_D_TOKEN else f'consider updating to {d_token_best_val}'}")
print(f"  • SSL (already decided): PROCEED — use pretrained init (mask_ratio=0.30 from 03_ssl_pretraining.ipynb)")